## AISE4010- Assignment3 - Time Series Classification using TCN and Transformer + Hyperparameter Tuning

## Grade: 100 points

### Instructions

#### Follow These Steps before submitting your assignment

1. Complete the notebook.

2. Make sure all plots have axis labels.

3. Once the notebook is complete, `Restart` your kernel by clicking 'Kernel' > 'Restart & Run All'.

4. Fix any errors until your notebook runs without any problems.

5. Submit one completed notebook for the group to OWL by the deadline.

6. Make sure to reference all external code and documentation used.

In [1]:
import pandas as pd
import numpy as np

### Dataset

The dataset is a sample of 46 satellite images, collected in 2006, located in southwestern France near Toulouse. It
is a 24 km × 24 km area and the dataset uses 3 output classes (2 available) for arable soil classification based on the following paper: https://arxiv.org/pdf/1811.10166.

You will be using helper functions below to prepare it for deep learning models.

In [2]:
# Call this helper method by passing in the names of the provided training and test sets' files.
def read_SITS_data(name_file):
    data = pd.read_table(name_file, sep=',', header=None)

    y_data = data.iloc[:,0]
    y = np.asarray(y_data.values, dtype='uint8')
    y[y>1] = 0

    polygonID_data = data.iloc[:,1]
    polygon_ids = polygonID_data.values
    polygon_ids = np.asarray(polygon_ids, dtype='uint16')

    X_data = data.iloc[:,2:]
    X = X_data.values
    X = np.asarray(X, dtype='float32')

    return  X, polygon_ids, y

In [3]:
def custom_feature_scaling(train, test):
    min_per = np.percentile(train, 2, axis=(0,1))
    max_per = np.percentile(train, 100-2, axis=(0,1))

    new_train = (train-min_per)/(max_per-min_per)
    new_test = (test-min_per)/(max_per-min_per)

    return new_train, new_test

### Question 1 - Data Preprocessing (15%)
- Q1.1 Call "read_SITS_data()" for the training set and store the results as X_train, polygon_ids_train, and y_train.
- Q1.2 Call "read_SITS_data()" above for the test set and store the results as X_test, polygon_ids_test, and y_test.
- Q1.3 Reshape the training and test sets.
  - Each set must be reshaped into a 3-D array. The first dimension will be the number of rows of the original set. The second dimension will be int(x / 3), where x is the number of columns of the original set and int() is a casting function. The third dimension will be 3 (number of channels).
- Q1.4 Call "custom_feature_scaling()" with the training and test sets. Save the results as the final sets for use.
- Q1.5 How many entries are in the training set? How many time steps are in each entry? How many features are there for each time step? How many labels for each entry?


In [4]:
#Store datasets into variables
train_file = 'train_dataset.csv'
test_file = 'test_dataset.csv'

#Q1.1 Call "read_SITS_data" for the training dataset and store.
X_train, polygon_ids_train, y_train = read_SITS_data(train_file)

#Q1.2 Call "read_SITS_data" for the test dataset and store.
X_test, polygon_ids_test, y_test = read_SITS_data(test_file)

#Print shape of datasets to verify
print(X_train.shape)        
print(polygon_ids_train.shape)
print(y_train.shape)
print(X_test.shape)           
print(polygon_ids_test.shape)
print(y_test.shape)



(260, 447)
(260,)
(260,)
(260, 447)
(260,)
(260,)


In [5]:
# Q1.3 Reshape the training and test sets into 3D arrays for time series analysis.
n_train, x_train = X_train.shape
n_test,  x_test  = X_test.shape

# Number of time steps
time_steps_train = int(x_train / 3)
time_steps_test  = int(x_test  / 3)

# Reshape: (samples, time_steps, channels=3)
X_train_3d = X_train.reshape(n_train, time_steps_train, 3)
X_test_3d  = X_test.reshape(n_test,  time_steps_test,  3)

print("X_train_3d shape:", X_train_3d.shape)  # (n_train, time_steps_train, 3)
print("X_test_3d shape:",  X_test_3d.shape)   # (n_test,  time_steps_test,  3)

X_train_3d shape: (260, 149, 3)
X_test_3d shape: (260, 149, 3)


In [6]:
# Q1.4 Apply custom feature scaling to the 3D arrays. These arrays are our final input datasets.
X_train_final, X_test_final = custom_feature_scaling(X_train_3d, X_test_3d)

# Print shapes of the final scaled datasets to verify
print("Scaled X_train_final shape:", X_train_final.shape)
print("Scaled X_test_final shape:",  X_test_final.shape)

Scaled X_train_final shape: (260, 149, 3)
Scaled X_test_final shape: (260, 149, 3)


*Write your Answer to Q1.5 here:*
- There are 260 Entries in the training set(from first value in shape)
- There are 149 time steps per entry (from second value in shape)
- There are 3 features per time stamp, Corresponding to the 3 channels (from third value in shape)
- There is 1 entry per label, stored in the y_train variable



### Question2 - Temporal Convolutional Network
- Q2.1 Create a Sequential model for classification. The model should have a TCN layer of size 64, a fully connected layer of size 256, a dropout of 0.3, and a fully connected output layer with Softmax activation (Hint: the logits axis should be on 0). Train the model using the provided dataset for 20 epochs. Use the batch_size of 32, and ADAM optimizer. Print the model summary.
- Q2.2 Train the model with the same parameters, print the model summary and evaluate the model's accuracy on the test set. Print the accuracy.
- Q2.3 Why do we use the Softmax activation on the output layer? In what scenarios does this contrast to using ReLU instead?


In [7]:
#Q2.1 Create sequential model for classification task.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tcn import TCN   # pip install keras-tcn or tcn
# y_train / y_test are integer labels 0 or 1

n_timesteps  = X_train_final.shape[1]   # 149
n_features   = X_train_final.shape[2]   # 3
n_classes    = 2                        # soil class 0 or 1

model = Sequential()
model.add(
    TCN(64, input_shape=(n_timesteps, n_features))   # TCN layer of size 64
)
model.add(Dense(256, activation='relu'))             # fully connected 256
model.add(Dropout(0.3))                              # dropout 0.3
model.add(Dense(n_classes, activation='softmax'))    # softmax output

# Compile with Adam optimizer
model.compile(
    optimizer=Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Print model summary (Q2.1 requirement)
model.summary()

# Train for 20 epochs, batch size 32
history = model.fit(
    X_train_final, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test_final, y_test),
    verbose=1
)


C:\Users\Callu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ tcn (TCN)                       │ (None, 64)             │       136,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 153,922 (601.26 KB)

 Trainable params: 153,922 (601.26 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 90ms/step - accuracy: 0.8472 - loss: 0.8287 - val_accuracy: 0.8808 - val_loss: 0.2783
Epoch 2/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.8818 - loss: 0.3460 - val_accuracy: 0.9154 - val_loss: 0.2455
Epoch 3/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.9005 - loss: 0.2463 - val_accuracy: 0.9231 - val_loss: 0.2236
Epoch 4/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.9204 - loss: 0.2501 - val_accuracy: 0.9192 - val_loss: 0.2094
Epoch 5/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.9445 - loss: 0.1581 - val_accuracy: 0.9269 - val_loss: 0.2371
Epoch 6/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9284 - loss: 0.1486 - val_accuracy: 0.9269 - val_loss: 0.2792
Epoch 7/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9578 - loss: 0.0862 - val_accuracy: 0.9269 - val_loss: 0.2675
Epoch 8/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9699 - loss: 0.0871 - val_accuracy: 0.9500 - val_loss: 0.2408


In [8]:
#Q2.2 Evaluate on the test set
test_loss, test_accuracy = model.evaluate(X_test_final, y_test, verbose=0)
print("Test accuracy:", test_accuracy)


Test accuracy: 0.9230769276618958


*Write your Answer to Q2.3 Here:*
We use softmax because this is a classification task with discrete classes. Softmax converts the output to a vlaue inbetween 0 and 1 which makes it easy to classify the networks predictions. ReLU outputs the max(0,x) as an output. Since this value is not normalized (I.E. the value is unbounded), we dont get probablilities in order to classify between the 2 classes.




### Question 3 - Transformer Model
- Q3.1 Create a transformer encoder block. It should use MultiHeadAttention for residual connection. The projection layers can be two Conv1D layers, based on number of feed forward dimensions and with kernel sizes of 1.
- Q3.2 Define the model. It should have 4 encoder blocks, each with 256 heads and feed forward dimensions of 4. Add a flatten layer, then a fully connected layer of size 2 and a fully connected output layer.
- Q3.3 Print the model summary, train the model using 50 epochs and a batch size of 32. Evaluate the model accuracy on the test set and print it.


In [9]:
from tensorflow.keras.layers import (
    Input, LayerNormalization, MultiHeadAttention,
    Conv1D, Add, Flatten, Dense
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Q3.1 – TRANSFORMER ENCODER

def transformer_encoder(inputs, num_heads, ff_dim):
    # LayerNorm + Self-Attention
    x = LayerNormalization(epsilon=1e-6)(inputs)
    attn_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=inputs.shape[-1]   # ensures output shape matches residual
    )(x, x)
    x = Add()([inputs, attn_output])  # residual connection

    # LayerNorm + Feed Forward (Conv1D for "projection")
    y = LayerNormalization(epsilon=1e-6)(x)
    y = Conv1D(filters=ff_dim, kernel_size=1, activation='relu')(y)
    y = Conv1D(filters=inputs.shape[-1], kernel_size=1)(y)
    out = Add()([x, y])  # second residual

    return out





In [10]:
# Q3.2 – MODEL DEFINITION
n_timesteps = X_train_final.shape[1]   # 149
n_features  = X_train_final.shape[2]   # 3
n_classes   = 2

inputs = Input(shape=(n_timesteps, n_features))
x = inputs

#  Corrected transformer settings:
num_heads = 4       
ff_dim = 256        

for _ in range(4):
    x = transformer_encoder(x, num_heads=num_heads, ff_dim=ff_dim)

x = Flatten()(x)
x = Dense(2, activation='relu')(x)
outputs = Dense(n_classes, activation='softmax')(x)

transformer_model = Model(inputs, outputs)

transformer_model.compile(
    optimizer=Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Print model summary
transformer_model.summary()



Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 149, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 149, 3)    │          6 │ input_layer_1[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 149, 3)    │        183 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 149, 3)    │          0 │ input_layer_1[0]… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 149, 3)    │          6 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 149, 256)  │      1,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 149, 3)    │        771 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 149, 3)    │          0 │ add[0][0],        │
│                     │                   │            │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 149, 3)    │          6 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 149, 3)    │        183 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 149, 3)    │          0 │ add_1[0][0],      │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 149, 3)    │          6 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 149, 256)  │      1,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 149, 3)    │        771 │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 149, 3)    │          0 │ add_2[0][0],      │
│                     │                   │            │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 149, 3)    │          6 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 149, 3)    │        183 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 149, 3)    │          0 │ add_3[0][0],      │
│                     │                   │            │ multi_head_atten

 Total params: 8,862 (34.62 KB)

 Trainable params: 8,862 (34.62 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Q3.3 – TRAINING & ACCURACY

history = transformer_model.fit(
    X_train_final, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_final, y_test),
    verbose=1
)

test_loss, test_acc = transformer_model.evaluate(X_test_final, y_test, verbose=0)
print("Test accuracy:", test_acc)


Epoch 1/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 8s 153ms/step - accuracy: 0.5063 - loss: 0.7368 - val_accuracy: 0.9077 - val_loss: 0.6707
Epoch 2/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - accuracy: 0.9373 - loss: 0.6541 - val_accuracy: 0.9231 - val_loss: 0.5559
Epoch 3/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 99ms/step - accuracy: 0.9238 - loss: 0.4997 - val_accuracy: 0.9231 - val_loss: 0.3071
Epoch 4/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - accuracy: 0.9036 - loss: 0.3200 - val_accuracy: 0.9231 - val_loss: 0.2961
Epoch 5/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 93ms/step - accuracy: 0.9335 - loss: 0.2744 - val_accuracy: 0.9231 - val_loss: 0.2900
Epoch 6/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 93ms/step - accuracy: 0.9281 - loss: 0.2661 - val_accuracy: 0.9231 - val_loss: 0.2635
Epoch 7/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 93ms/step - accuracy: 0.9349 - loss: 0.2448 - val_accuracy: 0.9231 - val_loss: 0.2812
Epoch 8/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 0.9324 - loss: 0.2609 - val_accuracy: 0.9231 - val_loss: 0.2559

### Question 4 - Hyperparameter Tuning
- Q4.1 Define a search space for the number of neurons in the fully connected layer that follows the flatten layer. The lower bound should be 2, the upper bound should be 16, and it should search every other value in between. Also have the tuner decide whether or not a dropout layer of 0.3 should be added after the aforementioned layer.
- Q4.2 Using GridSearch, search for the best hyperparameters with respect to accuracy over 50 epochs.
- Q4.3 Using the best hyperparameters, rebuild the model and print the model accuracy.

In [12]:
# Define Model builder to continue with question 4
def build_model(neurons, use_dropout):
    inputs = Input(shape=(n_timesteps, n_features))
    x = inputs

    # 4 transformer blocks (same as Q3)
    for _ in range(4):
        x = transformer_encoder(x, num_heads=4, ff_dim=256)

    x = Flatten()(x)
    x = Dense(neurons, activation='relu')(x)

    if use_dropout:
        x = Dropout(0.3)(x)

    outputs = Dense(2, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=Adam(),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Q4.1 — Search Space
neuron_space = list(range(2, 17, 2))    # 2,4,6,8,10,12,14,16
dropout_space = [True, False]

print("Neuron options:", neuron_space)
print("Dropout options:", dropout_space)


Neuron options: [2, 4, 6, 8, 10, 12, 14, 16]
Dropout options: [True, False]


In [13]:
# Q4.2 Grid Search
best_acc = -1
best_params = None

for n in neuron_space:
    for drop in dropout_space:
        print(f"Testing neurons={n}, dropout={drop}")

        model = build_model(n, drop)   # already defined in Q3

        model.fit(
            X_train_final, y_train,
            epochs=50,
            batch_size=32,
            verbose=0
        )

        _, acc = model.evaluate(X_test_final, y_test, verbose=0)
        print("Accuracy:", acc)

        if acc > best_acc:
            best_acc = acc
            best_params = (n, drop)

print("Best accuracy:", best_acc)
print("Best parameters:", best_params)



Testing neurons=2, dropout=True
Accuracy: 0.9230769276618958
Testing neurons=2, dropout=False
Accuracy: 0.9384615421295166
Testing neurons=4, dropout=True
Accuracy: 0.9192307591438293
Testing neurons=4, dropout=False
Accuracy: 0.9461538195610046
Testing neurons=6, dropout=True
Accuracy: 0.9576923251152039
Testing neurons=6, dropout=False
Accuracy: 0.9653846025466919
Testing neurons=8, dropout=True
Accuracy: 0.9230769276618958
Testing neurons=8, dropout=False
Accuracy: 0.9461538195610046
Testing neurons=10, dropout=True
Accuracy: 0.949999988079071
Testing neurons=10, dropout=False
Accuracy: 0.9192307591438293
Testing neurons=12, dropout=True
Accuracy: 0.9192307591438293
Testing neurons=12, dropout=False
Accuracy: 0.9115384817123413
Testing neurons=14, dropout=True
Accuracy: 0.9269230961799622
Testing neurons=14, dropout=False
Accuracy: 0.9230769276618958
Testing neurons=16, dropout=True
Accuracy: 0.9384615421295166
Testing neurons=16, dropout=False
Accuracy: 0.9461538195610046
Best accu

In [14]:
# Q4.3 — Rebuild & Evaluate Best Model
best_neurons, best_dropout = best_params

final_model = build_model(best_neurons, best_dropout)

final_model.fit(
    X_train_final, y_train,
    epochs=50,
    batch_size=32,
    verbose=1
)

_, final_acc = final_model.evaluate(X_test_final, y_test, verbose=0)
print("\nFinal model accuracy:", final_acc)

Epoch 1/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 10s 74ms/step - accuracy: 0.6573 - loss: 0.5862
Epoch 2/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.9309 - loss: 0.2697
Epoch 3/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.9329 - loss: 0.2244
Epoch 4/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.9210 - loss: 0.2225
Epoch 5/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.9169 - loss: 0.2028
Epoch 6/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.9107 - loss: 0.2653
Epoch 7/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.9449 - loss: 0.1591
Epoch 8/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.9098 - loss: 0.2063
Epoch 9/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.9437 - loss: 0.1705
Epoch 10/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.9330 - loss: 0.1941
Epoch 11/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.9420 - loss: 0.1685
Epoch 12/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.9658 - loss: 0.1286


### Question 6 - Discussion (5%)
- Q6.1 Indicate other hyperparameters relevant to transformers that can be tuned.
- Q6.2 What are the advantages and disadvantages of using GridSearch for finding optimal hyperparameters?


*Write your Answer to Q6.1 and Q6.2 Here:*
Q6.1: For Transformers other hyperparameters used or that could be used are:
- Number of attention heads: which would affect the number of attention mechanisms that operate in each layer
- FFN Size: Impacts model capacity.
- Number of Encoder Layers: More layers the deeper the model but higher computational cost.
- Training parameters like learning rate, batch size, dropout rates, ect
- Type of positional Encoding Hyper parameters
- Activation Function, Attention Dropout rate and FFN dropout rate

Q6.2:
Advantages
- Simple to implement: Easy to setup and reson about
- Deterministic: Produces reproducible results since it always checks the same parameter grid
- Tests every possible combination, guarenteeing the best. Which also makes it good for smaller search spaces.

Disadvantages:
- Computationally expensive: Costs grow exponentially based on # of hyper parameters and values per parameter
- Inefficient: Spends equal time on good and bad areas
- Not Scalable: Becomes impractical for large models like transformers where training is longer and computationally expensive.
- Rigid: You need to pick the fixed hyper parameters ahead of time, which can lead to missing the true optimal point.

